# Motion

Examples for the 6-DOF motion API.

A single `motion` service controls every stage on the machine. Each call selects a
stage with `Stage(motion=MotionStage.…)`, and positions are expressed as a
`SixDofMotionPosition` with up to six axes:

- **Translations** `x`, `y`, `z` — `DistanceMeasure` (microns, nanometers or millimeters)
- **Rotations** `rx`, `ry`, `rz` — `AngleMeasure` (degrees or radians)

Stages differ in which axes they expose. `ReadAvailableAxes` reports the axes a stage
supports, and every position carries the same information in its `supported_axes` field.
A move only acts on the axes you set, so you can drive one axis or several at once.

This notebook covers the **Top**, **Bottom**, and **Base** stages. For each it reads the
current position and demonstrates a relative move and an absolute move.


In [ ]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")
print(f"Python version: {sys.version}")


## Setup

Run this cell once. It imports everything the examples need, sets the instrument
connection target, and defines the helpers (`print_position`, position readers,
relative/absolute movers, and target builders) reused by every stage below.


In [ ]:
import math

from femtum_sdk.api import InstrumentApi
from femtum_sdk.api.contracts.v1.instruments.health_pb2 import HealthRequest
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import (
    DistanceMeasure,
    DistanceMeasureUnit,
    AngleMeasure,
    AngleMeasureUnit,
)
from femtum_sdk.api.contracts.v1.dto.motion_pb2 import Axis, MotionStage, SixDofMotionPosition
from femtum_sdk.api.contracts.v1.instruments.motion_pb2 import (
    ReadAvailableAxesResponse,
    Stage,
    MoveOrder,
    AllAtOnceMove,
    OrderedMove,
    ReadAvailableAxesRequest,
    ReadCurrentPositionRequest,
    MoveToAbsolutePositionRequest,
    MoveToRelativePositionRequest,
)

api_host = "localhost:5210"

NANOMETERS_PER_MICRON = 1000.0
POSITION_TOLERANCE_UM = 1.0
ARC_SECONDS_PER_DEGREE = 3600.0
ANGLE_TOLERANCE_ARCSEC = 1.0


def microns(value: float) -> DistanceMeasure:
    return DistanceMeasure(value=value, unit=DistanceMeasureUnit.Microns)


def degrees(value: float) -> AngleMeasure:
    return AngleMeasure(value=value, unit=AngleMeasureUnit.Degrees)


def arc_seconds(value: float) -> AngleMeasure:
    # The motion API speaks degrees/radians only, so arc seconds enter as degrees.
    return AngleMeasure(value=value / ARC_SECONDS_PER_DEGREE, unit=AngleMeasureUnit.Degrees)


def to_microns(distance: DistanceMeasure) -> float:
    if distance.unit == DistanceMeasureUnit.Nanometers:
        return distance.value / NANOMETERS_PER_MICRON
    return distance.value


def to_arc_seconds(angle: AngleMeasure) -> float:
    if angle.unit == AngleMeasureUnit.Radians:
        return math.degrees(angle.value) * ARC_SECONDS_PER_DEGREE
    elif angle.unit == AngleMeasureUnit.Degrees:
        return angle.value * ARC_SECONDS_PER_DEGREE
    elif angle.unit == AngleMeasureUnit.ArcSeconds:
        return angle.value
    else:
        raise ValueError(f"Unsupported angle unit: {AngleMeasureUnit.Name(angle.unit)}")

def to_degrees(angle: AngleMeasure) -> float:
    if angle.unit == AngleMeasureUnit.Radians:
        return math.degrees(angle.value)
    elif angle.unit == AngleMeasureUnit.Degrees:
        return angle.value
    elif angle.unit == AngleMeasureUnit.ArcSeconds:
        return angle.value / ARC_SECONDS_PER_DEGREE
    else:
        raise ValueError(f"Unsupported angle unit: {AngleMeasureUnit.Name(angle.unit)}")



def print_position(position: SixDofMotionPosition) -> None:
    rows = (
        (Axis.AXIS_X, "X", position.x.value, DistanceMeasureUnit.Name(position.x.unit)),
        (Axis.AXIS_Y, "Y", position.y.value, DistanceMeasureUnit.Name(position.y.unit)),
        (Axis.AXIS_Z, "Z", position.z.value, DistanceMeasureUnit.Name(position.z.unit)),
        (Axis.AXIS_RX, "RX", position.rx.value, AngleMeasureUnit.Name(position.rx.unit)),
        (Axis.AXIS_RY, "RY", position.ry.value, AngleMeasureUnit.Name(position.ry.unit)),
        (Axis.AXIS_RZ, "RZ", position.rz.value, AngleMeasureUnit.Name(position.rz.unit)),
    )
    supported_axes = set(position.supported_axes)
    supported_rows = [row for row in rows if row[0] in supported_axes]
    if not supported_rows:
        print("  (no supported axes reported)")
        return
    for _, label, value, unit in supported_rows:
        print(f"  {label:<2} = {value:>12.3f} {unit}")


def read_available_axes(api: InstrumentApi, stage: MotionStage) -> list[Axis]:
    response: ReadAvailableAxesResponse = api.motion.ReadAvailableAxes(
        ReadAvailableAxesRequest(stage=Stage(motion=stage))
    )
    return list(response.available_axes)


def read_current_position(api: InstrumentApi, stage: MotionStage) -> SixDofMotionPosition:
    response = api.motion.ReadCurrentPosition(
        ReadCurrentPositionRequest(stage=Stage(motion=stage))
    )
    return response.position

def demo_read(api: InstrumentApi, stage: MotionStage) -> None:
    print(f"=== {MotionStage.Name(stage)} ===")
    available_axes = read_available_axes(api, stage)
    print("Available axes:", [Axis.Name(axis) for axis in available_axes])
    print("Current position:")
    print_position(read_current_position(api, stage))
    print()

### Health Check

In [ ]:
# Quick connectivity check against the instrument host.
with InstrumentApi(host=api_host) as api:
    print(api.health.GetHealth(HealthRequest()))


## Read Positions


### Physical Stages

In [ ]:
with InstrumentApi(host=api_host) as api:
    ## Physical Stages
    demo_read(api, MotionStage.MOTION_STAGE_TOP)
    demo_read(api, MotionStage.MOTION_STAGE_BOTTOM)

    ## Composed Stages
    demo_read(api, MotionStage.MOTION_STAGE_BASE)


## Move Relative
⚠️ **Warning:** Before running a move, ensure the path is clear to prevent collisions or equipment damage.

### All At Once

In [ ]:
# The Bottom stage exposes AXIS_X, so this single-axis example nudges X by +100 microns
# and leaves every other axis untouched. Swap the stage/axis to drive a different one.
stage = MotionStage.MOTION_STAGE_BOTTOM
step_um = 100.0

with InstrumentApi(host=api_host) as api:
    before = read_current_position(api, stage)

    response = api.motion.MoveToRelativePosition(
        MoveToRelativePositionRequest(
            stage=Stage(motion=stage),
            delta=SixDofMotionPosition(x=microns(step_um)),
            order=MoveOrder(all_at_once=AllAtOnceMove()),
        )
    )

    after = read_current_position(api, stage)

    print("Before move:")
    print_position(before)
    print("After move:")
    print_position(after)

    expected_x_um = to_microns(before.x) + step_um
    assert abs(to_microns(response.position.x) - expected_x_um) <= POSITION_TOLERANCE_UM
    assert abs(to_microns(after.x) - expected_x_um) <= POSITION_TOLERANCE_UM


### Radial Axis

In [ ]:
# The Base stage exposes AXIS_RZ. This relative example reads RZ in arc seconds, nudges
# it by +5 arc seconds, reads again, then moves back by -5 arc seconds so the round trip
# ends where it started. The motion API speaks degrees/radians, so the arc_seconds /
# to_arc_seconds helpers convert on the way in and out.
stage = MotionStage.MOTION_STAGE_BASE
step_arcsec = 5.0

with InstrumentApi(host=api_host) as api:
    is_rz_supported = Axis.AXIS_RZ in read_available_axes(api, stage)
    if is_rz_supported:
        before = read_current_position(api, stage)
        print(f"RZ before:            {to_arc_seconds(before.rz):>10.3f} arcsec")

        api.motion.MoveToRelativePosition(
            MoveToRelativePositionRequest(
                stage=Stage(motion=stage),
                delta=SixDofMotionPosition(rz=arc_seconds(step_arcsec)),
                order=MoveOrder(all_at_once=AllAtOnceMove()),
            )
        )

        moved = read_current_position(api, stage)
        print(f"RZ after +{step_arcsec:g} arcsec:  {to_arc_seconds(moved.rz):>10.3f} arcsec")

        api.motion.MoveToRelativePosition(
            MoveToRelativePositionRequest(
                stage=Stage(motion=stage),
                delta=SixDofMotionPosition(rz=arc_seconds(-step_arcsec)),
                order=MoveOrder(all_at_once=AllAtOnceMove()),
            )
        )

        after = read_current_position(api, stage)
        print(f"RZ after moving back: {to_arc_seconds(after.rz):>10.3f} arcsec ({to_degrees(after.rz):>10.3f} deg)")

        assert abs(to_arc_seconds(moved.rz) - to_arc_seconds(before.rz) - step_arcsec) <= ANGLE_TOLERANCE_ARCSEC
        assert abs(to_arc_seconds(after.rz) - to_arc_seconds(before.rz)) <= ANGLE_TOLERANCE_ARCSEC
    else:
        print("AXIS_RZ is not supported on this stage")



### Ordered

In [ ]:
import time

# Ordered moves drive one axis after another, in the listed order. The Bottom stage
# exposes X and Y: Move A steps both out (+) as Y then X, Move B brings them back (-)
# as X then Y, so the round trip ends at the starting position.
stage = MotionStage.MOTION_STAGE_BOTTOM
step_um = 10000.0

with InstrumentApi(host=api_host) as api:
    start = read_current_position(api, stage)

    move_a = api.motion.MoveToRelativePosition(
        MoveToRelativePositionRequest(
            stage=Stage(motion=stage),
            delta=SixDofMotionPosition(x=microns(step_um), y=microns(step_um)),
            order=MoveOrder(ordered=OrderedMove(axis=[Axis.AXIS_Y, Axis.AXIS_X])),
        )
    )
    print("Move A:")
    print_position(move_a.position)

    time.sleep(1.5)  # Wait for human to see the move

    move_b = api.motion.MoveToRelativePosition(
        MoveToRelativePositionRequest(
            stage=Stage(motion=stage),
            delta=SixDofMotionPosition(x=microns(-step_um), y=microns(-step_um)),
            order=MoveOrder(ordered=OrderedMove(axis=[Axis.AXIS_X, Axis.AXIS_Y])),
        )
    )
    print("Move B:")
    print_position(move_b.position)

    after = read_current_position(api, stage)

    assert abs(to_microns(after.x) - to_microns(start.x)) <= POSITION_TOLERANCE_UM
    assert abs(to_microns(after.y) - to_microns(start.y)) <= POSITION_TOLERANCE_UM

## Move Absolute
⚠️ **Warning:** Before running a move, ensure the path is clear to prevent collisions or equipment damage.

### All At Once

In [ ]:
# Absolute moves send an axis to a fixed coordinate rather than a relative offset. We
# read the Bottom stage's current X, target 100 microns away, and drive only that axis.
stage = MotionStage.MOTION_STAGE_BOTTOM

with InstrumentApi(host=api_host) as api:
    before = read_current_position(api, stage)
    target_x_um = to_microns(before.x) + 100.0

    response = api.motion.MoveToAbsolutePosition(
        MoveToAbsolutePositionRequest(
            stage=Stage(motion=stage),
            target=SixDofMotionPosition(x=microns(target_x_um)),
            order=MoveOrder(all_at_once=AllAtOnceMove()),
        )
    )

    after = read_current_position(api, stage)

    print("Before move:")
    print_position(before)
    print("After move:")
    print_position(after)

    assert abs(to_microns(response.position.x) - target_x_um) <= POSITION_TOLERANCE_UM
    assert abs(to_microns(after.x) - target_x_um) <= POSITION_TOLERANCE_UM
